# Add `DataBound` Properties to GeoJSON

This notebook loads a GeoJSON `FeatureCollection`, calculates a bounding box from each feature's geometry, and stores the four coordinates in `properties.DataBound`.

The `DataBound` coordinate order is:

`[west, south, east, north]`

## 1. Load GeoJSON Input

The input path is resolved by looking for the project root, so the notebook can be run from VS Code or from a terminal with different working directories.

In [12]:
import json
from pathlib import Path

from shapely.geometry import shape


def find_project_root(start: Path) -> Path:
    """Find the workspace folder containing both package.json and src/."""
    for candidate in (start, *start.parents):
        if (candidate / "package.json").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the london-explorer project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_PATH = PROJECT_ROOT / "src" / "assets" / "cityParams" / "newcastle.json"
OUTPUT_PATH = INPUT_PATH
PRECISION = 6

with INPUT_PATH.open("r", encoding="utf-8") as file:
    geojson = json.load(file)

if geojson.get("type") != "FeatureCollection":
    raise ValueError("The input must be a GeoJSON FeatureCollection.")

features = geojson.get("features", [])
if not features:
    raise ValueError("The FeatureCollection does not contain any features.")

print(f"Loaded {INPUT_PATH}")
print(f"Features: {len(features)}")

Loaded c:\Users\tichen\OneDrive - Foster + Partners\Documents\Project\london-explorer\src\assets\cityParams\newcastle.json
Features: 1


## 2. Calculate One `DataBound` per Feature

GeoJSON coordinates are ordered as longitude, latitude. Shapely's `bounds` returns `(west, south, east, north)` for each geometry.

In [13]:
def calculate_databound(feature: dict, index: int) -> list[float]:
    """Calculate [west, south, east, north] from a feature geometry."""
    geometry = feature.get("geometry")
    if not geometry:
        raise ValueError(f"Feature {index} does not contain a geometry.")

    geometry_object = shape(geometry)
    if geometry_object.is_empty:
        raise ValueError(f"Feature {index} contains an empty geometry.")

    west, south, east, north = geometry_object.bounds
    return [
        round(west, PRECISION),
        round(south, PRECISION),
        round(east, PRECISION),
        round(north, PRECISION),
    ]


for index, feature in enumerate(features):
    databound = calculate_databound(feature, index)
    feature.setdefault("properties", {})["dataBound"] = databound

print(f"Calculated dataBound for {len(features)} features.")

Calculated dataBound for 1 features.


## 3. Review the Calculated Values

The values below show the `DataBound` property generated for each feature.

In [14]:
for index, feature in enumerate(features):
    feature_name = feature.get("properties", {}).get("name", f"feature {index}")
    print(f"{feature_name}: {feature['properties']['dataBound']}")

newcastle: [-1.86187, 54.871278, -1.340174, 55.085706]


## 4. Export the Updated GeoJSON

The original geometry and properties are preserved. Each feature receives or replaces `properties.DataBound`.

In [15]:
with OUTPUT_PATH.open("w", encoding="utf-8") as file:
    json.dump(geojson, file, indent=2)
    file.write("\n")

print(f"Saved updated GeoJSON to {OUTPUT_PATH}")

Saved updated GeoJSON to c:\Users\tichen\OneDrive - Foster + Partners\Documents\Project\london-explorer\src\assets\cityParams\newcastle.json


## 5. Verify the Output

Reload the exported file and confirm that every feature contains a four-value `DataBound` array.

In [16]:
with OUTPUT_PATH.open("r", encoding="utf-8") as file:
    exported_geojson = json.load(file)

for index, feature in enumerate(exported_geojson["features"]):
    databound = feature.get("properties", {}).get("dataBound")
    if not isinstance(databound, list) or len(databound) != 4:
        raise ValueError(f"Feature {index} has an invalid DataBound: {databound!r}")

print(f"Verified DataBound on {len(exported_geojson['features'])} exported features.")

Verified DataBound on 1 exported features.


## 6. Example Output Shape

For a feature, the generated property has this form:

```json
"properties": {
  "DataBound": [west, south, east, north]
}
```

In [17]:
print(json.dumps(
    exported_geojson["features"][0]["properties"]["dataBound"],
    indent=2,
))

[
  -1.86187,
  54.871278,
  -1.340174,
  55.085706
]
